# Deployment Lab — AI Lab Instructor Notebook

*MLOps & Systems · Expert*

Serve a tiny deterministic model over HTTP, then validate the service contract from a client.

**Outcome.** Students can expose a model as a minimal JSON API, implement health and prediction endpoints, and verify deployment behavior with contract tests.

5 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Deployment Lab — Student

Offline lab: serve a tiny model via HTTP and write contract tests.

## Task 0.0 — Setup

Import the stdlib HTTP tools, NumPy, and the shared `check(...)` helper. Later sections use this setup to run both the server and the client in one notebook.

In [ ]:
import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** # Deployment Lab — Instructor

Offline lab: serve a tiny model via HTTP and write contract tests.

## 0 — Deterministic model stub

## Task 2: 0 — Deterministic model stub

## 0 — Deterministic model stub

Implement a tiny offline model with a stable contract.

- `sigmoid(z)`: convert logits to probabilities
- `predict_one(x)`: validate input shape, compute probability, and threshold to a label

This model stub becomes the logic behind the HTTP service in the next section.

In [ ]:
D = 8
W = np.linspace(-1.0, 1.0, D).astype(np.float64)
B = 0.1

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def predict_one(x: np.ndarray):
    x = np.asarray(x, dtype=np.float64)
    if x.shape != (D,):
        raise ValueError(f"expected shape ({D},) got {x.shape}")
    p = float(sigmoid(float(x @ W + B)))
    y = int(p >= 0.5)
    return p, y

x_demo = rng.standard_normal((D,))
p_demo, y_demo = predict_one(x_demo)
print('demo', p_demo, y_demo)
check('p_range', 0.0 <= p_demo <= 1.0)


In [ ]:
# Checks
check('p_range', 0.0 <= p_demo <= 1.0)

## Task 3: 1 — HTTP service contract (browser-safe stdlib)

## 1 — HTTP service contract (browser-safe stdlib)

Expose the deterministic model through a minimal JSON API using only the Python standard library.

- implement a browser-safe local service with `GET /health` and `POST /predict`
- return JSON for both success and error cases
- in the platform runtime, simulate the HTTP contract locally instead of opening a listening socket

At the end of the cell, print the base URL and verify that the health endpoint responds.

In [ ]:
$f

In [ ]:
# Checks
check('base_url_http', BASE_URL.startswith('http://'))
check('server_health_200', health_code == 200)
check('server_health_ok', health_obj.get('status') == 'ok')


## Task 4: 2 — Client helpers

## 2 — Client helpers

Implement thin client helpers for JSON GET and POST requests.

- `http_get_json(url)`: return `(status_code, parsed_json)`
- `http_post_json(url, obj)`: send JSON and return `(status_code, parsed_json)`
- support the notebook's local service contract first, then fall back to real URL requests when available
- on HTTP errors, return the status code together with the parsed error payload

These helpers are then reused by the contract tests.

In [ ]:
$10

In [ ]:
# Checks
check('helper_health', http_get_json(BASE_URL + '/health')[0] == 200)
check('helper_bad_request', http_post_json(BASE_URL + '/predict', {'x': [1,2,3]})[0] == 400)


## Task 5: 3 — Contract tests (CI-style)

## 3 — Contract tests (CI-style)

Write the deployment-facing tests that validate the service contract.

- health endpoint returns 200 with the expected schema
- valid prediction requests return probability and binary label
- malformed requests return a controlled 400 error

Run all three tests as if they were CI checks against a deployed service.

In [ ]:
def test_health():
    code, obj = http_get_json(BASE_URL + "/health")
    check("health_200", code == 200)
    check("health_schema", obj.get("status") == "ok")

def test_predict_ok():
    x = rng.standard_normal((D,)).tolist()
    code, obj = http_post_json(BASE_URL + "/predict", {"x": x})
    check("predict_200", code == 200)
    check("predict_has_proba", "proba" in obj)
    check("predict_has_label", "label" in obj)
    check("proba_range", 0.0 <= float(obj["proba"]) <= 1.0)
    check("label_binary", int(obj["label"]) in (0, 1))

def test_predict_bad_request():
    code, obj = http_post_json(BASE_URL + "/predict", {"x": [1,2,3]})
    check("bad_400", code == 400)
    check("bad_has_error", "error" in obj)

test_health()
test_predict_ok()
test_predict_bad_request()
print('all tests passed')

check('contract_tests_ran', True)


In [ ]:
# Checks
check('contract_tests_ran', True)